# 05 Classification: High-Price Listing Potential

Objective: classify whether a listing belongs to the high-price segment.

In [ ]:
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import confusion_matrix, classification_report
import joblib

from src.modeling import build_preprocessor
from src.evaluation import classification_metrics

DATA_PROCESSED = Path("../data/processed")
MODELS_DIR = Path("../models")

df = pd.read_csv(DATA_PROCESSED / "modeling_airbnb_athens.csv")
df.head()

In [ ]:
target = "high_price_listing"

numeric_features = [
    col for col in [
        "accommodates", "bathrooms", "bedrooms", "beds", "amenities_count",
        "minimum_nights", "availability_365", "availability_rate",
        "number_of_reviews", "review_scores_rating", "reviews_per_month",
        "host_experience_days"
    ] if col in df.columns
]

categorical_features = [
    col for col in [
        "neighbourhood_cleansed", "property_type", "room_type",
        "host_is_superhost", "instant_bookable", "is_entire_home"
    ] if col in df.columns
]

model_df = df[numeric_features + categorical_features + [target]].dropna(subset=[target])
X = model_df[numeric_features + categorical_features]
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
preprocessor = build_preprocessor(numeric_features, categorical_features)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

results = []

for name, model in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)

    if hasattr(pipe, "predict_proba"):
        proba = pipe.predict_proba(X_test)[:, 1]
    else:
        proba = None

    metrics = classification_metrics(y_test, preds, proba)
    metrics["Model"] = name
    results.append(metrics)

results_df = pd.DataFrame(results).sort_values("ROC_AUC", ascending=False)
results_df

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]

best_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", best_model)
])

best_pipe.fit(X_train, y_train)
joblib.dump(best_pipe, MODELS_DIR / "best_high_price_classifier.pkl")

best_model_name